In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.feature_selection import SelectKBest, mutual_info_regression
from sklearn.base import BaseEstimator, TransformerMixin

from sklearn.ensemble import RandomForestRegressor



In [5]:
df = pd.read_csv("santander-value-prediction-challenge/train.csv")


In [7]:
df.shape

(4459, 4993)

In [9]:
df.describe()

,target,48df886f9,0deb4b6a8,34b15f335,a8cb14b00,2f0771a37,30347e683,d08d1fbe3,6ee66e115,20aa07010,...,3ecc09859,9281abeea,8675bec0b,3a13ed79a,f677d4d13,71b203550,137efaa80,fb36b89d9,7e293fbaf,9fc776466
count,4.459000e+03,4.459000e+03,4.459000e+03,4.459000e+03,4.459000e+03,4.459000e+03,4.459000e+03,4.459000e+03,4.459000e+03,4.459000e+03,...,4.459000e+03,4.459000e+03,4.459000e+03,4.459000e+03,4459.000000,4.459000e+03,4.459000e+03,4.459000e+03,4.459000e+03,4.459000e+03
mean,5.944923e+06,1.465493e+04,1.390895e+03,2.672245e+04,4.530164e+03,2.640996e+04,3.070811e+04,1.686522e+04,4.669208e+03,2.569407e+06,...,4.676057e+05,4.446239e+05,8.056219e+05,7.812966e+05,143.529939,1.213809e+05,3.573451e+04,3.123741e+05,9.219960e+04,2.279100e+05
std,8.234312e+06,3.893298e+05,6.428302e+04,5.699652e+05,2.359124e+05,1.514730e+06,5.770590e+05,7.512756e+05,1.879449e+05,9.610183e+06,...,4.068038e+06,4.428889e+06,4.513246e+06,6.839451e+06,9584.318507,4.720709e+06,1.614622e+06,4.318501e+06,1.635993e+06,1.811139e+06
min,3.000000e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,6.000000e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
50%,2.260000e+06,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
75%,8.000000e+06,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,6.000000e+05,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
max,4.000000e+07,2.000000e+07,4.000000e+06,2.000000e+07,1.480000e+07,1.000000e+08,2.070800e+07,4.000000e+07,1.040000e+07,3.196120e+08,...,7.600000e+07,1.235880e+08,1.300000e+08,1.444000e+08,640000.000000,3.013120e+08,1.064200e+08,1.400000e+08,6.176800e+07,4.320000e+07


In [10]:
target_col = "target"
y = df[target_col]
x = df.drop(columns=[target_col, "ID"])

cat_cols = x.select_dtypes(include=["object"]).columns.to_list()
num_cols = x.select_dtypes(include=["int64", "float64"]).columns.to_list()


In [11]:
X_train, X_valid, y_train, y_valid = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("y_train:", y_train.shape)
print("y_valid:", y_valid.shape)

X_train: (3567, 4991)
X_valid: (892, 4991)
y_train: (3567,)
y_valid: (892,)


In [12]:
variance_selector = VarianceThreshold(threshold=0.01)

X_train_var = variance_selector.fit_transform(X_train)
X_valid_var = variance_selector.transform(X_valid)

print(X_train_var.shape)
print(X_valid_var.shape)

(3567, 4669)
(892, 4669)


In [13]:
X_train_var = pd.DataFrame(X_train_var, columns=X_train.columns[variance_selector.get_support()])

X_valid_var = pd.DataFrame(X_valid_var, columns=X_train.columns[variance_selector.get_support()])

In [14]:
class CorrelationFilter(BaseEstimator, TransformerMixin):

    def __init__(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        corr_matrix = X.corr().abs()

        upper = corr_matrix.where(
            np.triu(
                np.ones(corr_matrix.shape),
                k=1
            ).astype(bool)
        )

        self.features_to_drop_ = [
            column
            for column in upper.columns
            if any(upper[column] > self.threshold)
        ]

        self.features_to_keep_ = [
            column
            for column in X.columns
            if column not in self.features_to_drop_
        ]

        return self

    def transform(self, X):
        X = pd.DataFrame(X)

        return X.drop(
            columns=self.features_to_drop_,
            errors="ignore"
        )

In [15]:
corr_filter = CorrelationFilter(threshold=0.95)

X_train_corr = corr_filter.fit_transform(X_train_var)
X_valid_corr = corr_filter.transform(X_valid_var)

In [18]:
mi_filter = SelectKBest(
    score_func=mutual_info_regression,
    k=200
)

X_train_kbest = mi_filter.fit_transform(
    X_train_corr,
    y_train
)

X_valid_kbest = mi_filter.transform(
    X_valid_corr
)

In [19]:
selected_features = X_train_corr.columns[
    mi_filter.get_support()
]

print("Number of selected features:", len(selected_features))
print("Selected features:")
print(selected_features.tolist())

Number of selected features: 200
Selected features:
['20aa07010', '22ed6dba3', '87ffda550', '822e49b95', '251d1aa17', '963a49cdc', 'ad207f7bb', '68a945b18', '0e1f6696a', '63c094ba4', 'b30e932ba', '935ca66a9', 'a6e871369', '6d2ece683', 'e078302ef', '861076e21', '6c0e0801a', 'bee629024', '26fc93eb7', '0572565c2', '66ace2992', '350473311', 'ac30af84a', '0c9462c08', 'fb49e4212', '6619d81fc', 'bdf773176', '1857fbccf', '8bd53906a', 'aca228668', 'd47c58fe2', '580f5ff06', '3c8a3ced0', '6eef030c1', 'bc70cbc26', 'fc99f9426', '070f95c99', '7e814a30d', 'be2e15279', '6f88afe65', 'c7525612c', '1c71183bb', '375c6080e', 'bd6da0cca', 'fbaed5913', '93ca30057', 'f3cf9341c', '1db387535', 'b98f3e0d7', 'b6c0969a2', 'f16a196c6', '8c0a1fa32', 'ce3d7595b', 'b43a7cfd5', '024c577b9', '4d2671746', 'aac52d8d9', 'ea772e115', 'ad009c8b9', 'b85fa8b27', '341daa7d1', 'b4cfe861f', '51d4053c7', '9c42bff81', '0a953f97e', 'd428161d9', '64c6eb1cb', '2ec5b290f', '9306da53f', '44d5b820f', '5f341a818', '9884166a7', '1fa0f78d0'

In [36]:
X_train_kbest = pd.DataFrame(
    X_train_kbest,
    columns=selected_features,
    index=X_train_kbest.index
)

X_valid_kbest = pd.DataFrame(
    X_valid_kbest,
    columns=selected_features,
    index=X_valid_kbest.index
)

In [23]:
feature_scores = pd.DataFrame({
    "feature": X_train_corr.columns,
    "score": mi_filter.scores_,
})

feature_scores = feature_scores.sort_values(
    by="score",
    ascending=False
)

print(feature_scores)

        feature     score
3936  f190486d6  0.123254
629   6619d81fc  0.115273
4243  190db8488  0.115085
612   fb49e4212  0.114320
2290  58e2e02e6  0.113603
...         ...       ...
1     0deb4b6a8  0.000000
2245  a2d1008bb  0.000000
2240  7b922ea8b  0.000000
2239  42ed6824a  0.000000
4491  9fc776466  0.000000

[4492 rows x 2 columns]


In [24]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train_kbest,
    y_train
)

,n_estimators,300
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [25]:
y_pred = model.predict(X_valid_kbest)

c:\Users\zand\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [26]:
from sklearn.metrics import mean_absolute_error

mae = mean_absolute_error(
    y_valid,
    y_pred
)

print("MAE:", mae)

MAE: 4645142.195385027


In [28]:
feature_importance = pd.Series(
    model.feature_importances_,
    index=X_train_kbest.columns
).sort_values(ascending=False)

print(feature_importance)

f190486d6    0.093863
58e2e02e6    0.036629
eeb9cd3aa    0.034360
15ace8c9f    0.025332
b43a7cfd5    0.023387
               ...   
fbaed5913    0.000478
b2e82c050    0.000435
93ca30057    0.000403
6b119d8ce    0.000397
88d29cfaf    0.000385
Length: 200, dtype: float64


In [46]:
X_train_kbest = pd.DataFrame(
    X_train_kbest,
    columns=selected_features,
)

X_valid_kbest = pd.DataFrame(
    X_valid_kbest,
    columns=selected_features,
)

In [47]:
for k in [50, 100, 150]:

    selected = feature_importance.head(k).index

    X_train_rf = X_train_kbest[selected]
    X_test_rf = X_valid_kbest[selected]

 

    model.fit(X_train_rf, y_train)

    y_pred = model.predict(X_test_rf)

    mae = mean_absolute_error(y_valid, y_pred)

    print(f"{k} features -> MAE: {mae}")

50 features -> MAE: 4984716.410467507
100 features -> MAE: 4991912.707054591
150 features -> MAE: 4982318.37886286
